# Block 04: Synthetic Theorem-Regime Verification

**Goal**  
Use controlled QPSimulator families to verify the rank-1 theorem regime, CRB variance law, and energy-resolution scaling cleanly.

**Checklist alignment**  
Implements E1, E4, and the simulation side of E5.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [ ]:
out = run_block04_theorem_suite(cfg)
display(out["rank1_summary_df"])
display(out["crb_df"])
display(out["resolution_df"])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df = out["resolution_df"]
ax.loglog(df["noise_power"], df["sigma_emp"], marker="o", label="empirical")
ax.loglog(df["noise_power"], df["sigma_pred"], marker="s", label="predicted")
ax.set_xlabel("noise power")
ax.set_ylabel("sigma_E proxy")
ax.set_title("Simulation energy-resolution scaling")
ax.legend()
save_figure(fig, dirs["figures"] / "block_04_e5_resolution_scaling.png")
plt.close(fig)

In [ ]:
rank1_path = dirs["tables"] / "block_04_e1_rank1_summary.csv"
crb_path = dirs["tables"] / "block_04_e4_crb.csv"
resolution_path = dirs["tables"] / "block_04_e5_resolution.csv"
manifest_path = dirs["manifests"] / "block_04_theorem_summary.json"
save_dataframe(out["rank1_summary_df"], rank1_path)
save_dataframe(out["crb_df"], crb_path)
save_dataframe(out["resolution_df"], resolution_path)
save_json(out["resolution_summary"], manifest_path)
pd.DataFrame(
    [
        manifest_row("block_04", "theorem-support", str(rank1_path.relative_to(REPO_ROOT)), cfg),
        manifest_row("block_04", "theorem-support", str(crb_path.relative_to(REPO_ROOT)), cfg),
    ]
)